### Using built-in ase optimization with vasp



ASE has some nice optimization tools built into it. We can use them in vasp too. This example is adapted from this test: [https://trac.fysik.dtu.dk/projects/ase/browser/trunk/ase/test/vasp/vasp_Al_volrelax.py](https://trac.fysik.dtu.dk/projects/ase/browser/trunk/ase/test/vasp/vasp_Al_volrelax.py)

First the VASP way.



In [10]:
from vasp import Vasp
from ase.build import bulk

Al = bulk('Al', 'fcc', a=4.5, cubic=True)
calc = Vasp(label='bulk/Al-lda-vasp',
            xc='LDA', isif=7, nsw=5,
            ibrion=1, ediffg=-1e-3,
            lwave=False, lcharg=False,
            atoms=Al)
print(calc.potential_energy)
print(calc)

-10.07430731
VASP Calculation: /home/jovyan/dft-book/notebooks/04-bulk-systems/bulk/Al-lda-vasp

Formula: Al4
Number of atoms: 4

Unit cell:
  Volume: 73.3685 Å³
  a=4.1864 b=4.1864 c=4.1864 Å
  α=90.00° β=90.00° γ=90.00°

Energy: -10.074307 eV (-2.518577 eV/atom)
Max force: 0.000000 eV/Å
Pressure: -1.533 GPa

Parameters:
  xc: LDA
  kpts: (1, 1, 1)
  ismear: 1
  sigma: 0.1
  ibrion: 1
  isif: 7
  nsw: 5


Now, the ASE way.




In [ ]:
from vasp import Vasp
from ase.build import bulk
from ase.optimize import BFGS as QuasiNewton
from ase.constraints import StrainFilter
import os
import numpy as np

Al = bulk('Al', 'fcc', a=4.5, cubic=True)

calc = Vasp(label='bulk/Al-lda-ase',
            xc='LDA',
            atoms=Al)

# Check if we have a completed calculation to work with
if os.path.exists(os.path.join(calc.directory, 'OUTCAR')):
    # Get initial energy to ensure calculator is properly attached
    Al.get_potential_energy()
    
    sf = StrainFilter(Al)
    qn = QuasiNewton(sf, logfile='relaxation.log')
    try:
        qn.run(fmax=0.1, steps=5)
        print('Stress:\n', calc.stress)
        print('Al post ASE volume relaxation\n', calc.load_atoms().get_cell())
        print(calc)
    except np.linalg.LinAlgError as e:
        print(f'Optimization failed: {e}')
        print('This can happen when stress values cause numerical instability in BFGS.')
        print('The VASP internal optimizer (ISIF=7) in the previous cell is more robust.')
else:
    print('VASP calculation not yet complete. Run previous cell first.')
    print('Note: ASE StrainFilter optimization requires VASP to compute stress.')

Now for a detailed comparison:



In [ ]:
from vasp import Vasp
import os
import numpy as np

calc1 = Vasp(label='bulk/Al-lda-vasp')
calc2 = Vasp(label='bulk/Al-lda-ase')

# Check if both calculations exist
if (os.path.exists(os.path.join(calc1.directory, 'OUTCAR')) and 
    os.path.exists(os.path.join(calc2.directory, 'OUTCAR'))):
    atoms = calc1.load_atoms()
    atoms2 = calc2.load_atoms()

    cellA = atoms.get_cell()
    cellB = atoms2.get_cell()

    print((np.abs(cellA - cellB) < 0.01).all())
else:
    print('One or both calculations not complete. Run previous cells first.')

This could be handy if you want to use any of the optimizers in mod:ase.optimize in conjunction with mod:ase.constraints, which are more advanced than what is in VASP.

